In [1]:
import meshio
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

msh_file = Path(r"E:\ADATA\LITHIUM\OGS\Jupyter notebooks\mesh5.msh")
vtu_file = input_dir / "mesh5.vtu"

mesh = meshio.read(msh_file)
meshio.write(vtu_file, mesh)

print("Created:", vtu_file)
print("Cell data keys:", mesh.cell_data_dict.keys())

for key in mesh.cell_data_dict.keys():
    print("\nKEY:", key)
    for ctype, data in mesh.cell_data_dict[key].items():
        print(ctype, sorted(set(data))[:30])


Created: E:\ADATA\LITHIUM\OGS\OGS_files\input\mesh5.vtu
Cell data keys: dict_keys(['gmsh:physical', 'gmsh:geometrical'])

KEY: gmsh:physical
triangle [np.int32(301), np.int32(302), np.int32(303), np.int32(304), np.int32(305), np.int32(306), np.int32(401), np.int32(402), np.int32(403), np.int32(404)]
tetra [np.int32(101), np.int32(102), np.int32(103), np.int32(104), np.int32(105), np.int32(201), np.int32(202)]

KEY: gmsh:geometrical
triangle [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(6), np.int32(9), np.int32(11), np.int32(13), np.int32(14), np.int32(15), np.int32(17), np.int32(22), np.int32(23), np.int32(24), np.int32(26), np.int32(27), np.int32(28), np.int32(31), np.int32(32), np.int32(33), np.int32(34), np.int32(36), np.int32(37), np.int32(38), np.int32(39), np.int32(40)]
tetra [np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14)]


In [2]:
import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

bulk = meshio.read(input_dir / "mesh5.vtu")

tetra = bulk.cells_dict["tetra"]
phys_tetra = bulk.cell_data_dict["gmsh:physical"]["tetra"]

for tag, name in [(201, "well1"), (202, "well2")]:
    selected = tetra[phys_tetra == tag]

    used_points = np.unique(selected.flatten())
    old_to_new = {old: new for new, old in enumerate(used_points)}

    new_points = bulk.points[used_points]

    new_tetra = np.array(
        [[old_to_new[node] for node in elem] for elem in selected],
        dtype=np.int64
    )

    bulk_node_ids = used_points.astype(np.uint64)

    submesh = meshio.Mesh(
        points=new_points,
        cells=[("tetra", new_tetra)],
        point_data={"bulk_node_ids": bulk_node_ids},
        cell_data={
            "gmsh:physical": [np.full(len(new_tetra), tag, dtype=np.int32)]
        }
    )

    out = input_dir / f"{name}.vtu"
    meshio.write(out, submesh)

    print(f"Created {out.name}")
    print("  tetra cells:", len(new_tetra))
    print("  points:", len(new_points))
    print("  bulk_node_ids dtype:", bulk_node_ids.dtype)

Created well1.vtu
  tetra cells: 194
  points: 91
  bulk_node_ids dtype: uint64
Created well2.vtu
  tetra cells: 194
  points: 91
  bulk_node_ids dtype: uint64
